# ブートストラップ法で評価値の信頼区間を求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_evaluation/bootstrap_ci.ipynb)

NLP クックブック 13-09 ／ 評価指標と統計的検定

テストセットで正解率 85% だったが、その値がどれくらい不確かなのか、信頼区間を付けて報告したい。

In [1]:
import numpy as np

rng = np.random.default_rng(0)
n = 500
gold = rng.integers(0, 2, n)
pred = np.where(rng.random(n) < 0.85, gold, 1 - gold)      # 正解率が約 85% の予測を人工的に作る

def bootstrap_ci(metric, gold, pred, n_boot=2000, alpha=0.05):
    """テスト事例を復元抽出して指標を計算し直すことを繰り返し、分布の両端を信頼区間とする"""
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(gold), len(gold))
        stats.append(metric(gold[idx], pred[idx]))
    return metric(gold, pred), np.percentile(stats, 100 * alpha / 2), np.percentile(stats, 100 * (1 - alpha / 2))

accuracy = lambda g, p: (g == p).mean()
point, lo, hi = bootstrap_ci(accuracy, gold, pred)
print(f"正解率 = {point:.3f}  95% 信頼区間 [{lo:.3f}, {hi:.3f}]")

正解率 = 0.862  95% 信頼区間 [0.832, 0.892]


In [2]:
# F1 のように単純な平均でない指標でも、同じ手順で求められる
from sklearn.metrics import f1_score

point, lo, hi = bootstrap_ci(lambda g, p: f1_score(g, p), gold, pred)
print(f"F1 = {point:.3f}  95% 信頼区間 [{lo:.3f}, {hi:.3f}]")

# テストセットの大きさと信頼区間の幅
for size in [100, 500, 2000, 10000]:
    g = rng.integers(0, 2, size); p = np.where(rng.random(size) < 0.85, g, 1 - g)
    _, lo, hi = bootstrap_ci(accuracy, g, p, n_boot=500)
    print(f"n = {size:5}: 幅 ±{(hi - lo) / 2 * 100:.1f} ポイント")

F1 = 0.873  95% 信頼区間 [0.842, 0.902]
n =   100: 幅 ±7.0 ポイント
n =   500: 幅 ±3.0 ポイント
n =  2000: 幅 ±1.6 ポイント
n = 10000: 幅 ±0.6 ポイント


- 500 件のテストセットでは、正解率に ±3 ポイント程度の不確かさがある。1 ポイントの差を議論するには数千件が必要。
- BLEU のようなコーパス単位の指標も、文をリサンプリングして計算し直せば同じように扱える (`sacrebleu` にも実装がある)。
- これは「テストセットの選び方」による不確かさ。学習の乱数シードによるばらつきは別に評価する必要がある。